# 리밸런싱 전략 모의투자 실행 노트북

`id=23 균형 멀티팩터(PIT KOSPI200·분기)` 등 리밸런싱 전략을 **KIS 모의투자(vts)** 로 점검·실행한다.
`RebalanceRunner` 를 그대로 재사용하므로 백테스트와 동일한 선정·목표비중 로직을 탄다.

## 실행 환경
이 노트북은 **`web` 컨테이너의 app 환경**(KRX/OpenDART/KIS 자격증명이 주입된)에서 실행해야 한다.
이미지에 jupyter 가 없다면 아래 중 하나로 실행한다.

```bash
# (A) 미리보기만 — CLI 스크립트(주문 안 냄)
docker compose run --rm web python scripts/paper_rebalance.py --strategy 23
# (B) 모의투자 실행
docker compose run --rm web python scripts/paper_rebalance.py --strategy 23 --execute
# (C) 이 노트북을 그대로 실행하려면 컨테이너에 jupyter 설치 후 기동
docker compose run --rm -p 8888:8888 web sh -lc 'pip install jupyterlab && jupyter lab --ip 0.0.0.0 --allow-root --NotebookApp.token=""'
```

> ⚠️ **안전장치**: 실행 셀은 기본 `EXECUTE=False` 이고, `settings.is_paper_trading`(KIS_ENV=vts)가 아니면 실행을 거부한다.

In [ ]:
import os, sys

# 'app' 패키지를 담은 backend/ 를 sys.path 에 올린다(노트북 실행 위치 무관).
ROOT = os.getcwd()
while ROOT != os.path.dirname(ROOT) and not os.path.isdir(os.path.join(ROOT, 'app')):
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from app.core.config import settings
from app.core.redis import redis_client
from app.services.market import now_kst
from engine.rebalance_runner import RebalanceRunner

STRATEGY_ID = 23  # 구동할 전략 ID
print('backend 경로:', ROOT)
print('모의투자 모드(is_paper_trading):', settings.is_paper_trading, '| KIS_ENV:', settings.KIS_ENV)

## 1) 미리보기 — 지금 리밸런싱하면 낼 주문 (주문 안 냄)

PIT 후보풀(KOSPI200 현재 구성) → 상대강도 상위 pick 축소 → 종합점수 선정 → 목표비중 → 산출 주문을 계산한다. **실제 매매는 하지 않는다.**

In [ ]:
runner = RebalanceRunner(STRATEGY_ID, redis_client)
plan = await runner.preview()  # Jupyter/IPython 는 top-level await 지원

print(f"기준일 {plan['as_of']} | 후보풀 {plan['pool_size']}종목 | 레짐 위험회피 {plan['risk_off']}")
print(f"목표비중 {len(plan['targets'])}종목:")
for sym, w in sorted(plan['targets'].items(), key=lambda x: -x[1]):
    px = plan['prices'].get(sym)
    print(f"  {sym}  {w*100:5.1f}%" + (f"  @ {px:,.0f}" if px else '  (현재가 미조회)'))
print('현재 보유:', plan['positions'] or '(없음)')
print('산출 주문:')
for sym, side, qty in plan['orders']:
    print(f"  {side.upper():4} {sym} x{qty}")

## 2) 모의투자 실행 (선택)

위 미리보기 주문이 마음에 들면 `EXECUTE = True` 로 바꾸고 이 셀을 실행한다. KIS 모의투자 계좌로 실제 주문이 전송된다.
수동 강제 1회 실행이라 장중/발화 스케줄과 무관하며 Redis 의 '마지막 실행일'은 소비하지 않는다.

In [ ]:
EXECUTE = False  # ← 실제 모의투자 주문을 내려면 True 로 바꾼다

assert settings.is_paper_trading, '모의투자(KIS_ENV=vts)가 아니면 실행 금지 — 실전 오발주 방지'
if EXECUTE and plan['orders']:
    await runner._rebalance_once(now_kst(), risk_off=plan['risk_off'], bar_tag='manual')
    print('주문 전송 완료 — 앱의 체결 로그/모의투자 계좌에서 확인하세요.')
else:
    print('EXECUTE=False 또는 낼 주문 없음 — 아무 것도 전송하지 않았습니다.')

In [ ]:
# 정리: Redis 연결 닫기(노트북 종료 전 1회)
await redis_client.aclose()